# NCAA Bracket Model — Modeling

In [298]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, log_loss, brier_score_loss



In [299]:
pd.set_option('display.max_rows', 100)

In [300]:
X_train = pd.read_parquet('../data/processed/X_train.parquet', engine='fastparquet')
X_val = pd.read_parquet('../data/processed/X_val.parquet', engine='fastparquet')
X_test = pd.read_parquet('../data/processed/X_test.parquet', engine='fastparquet')

y_train = pd.read_parquet('../data/processed/y_train.parquet', engine='fastparquet').squeeze()
y_val = pd.read_parquet('../data/processed/y_val.parquet', engine='fastparquet').squeeze()
y_test = pd.read_parquet('../data/processed/y_test.parquet', engine='fastparquet').squeeze()


assert X_train.shape[1] == X_val.shape[1] == X_test.shape[1], "Feature count mismatch across splits"
print(f"Shapes confirmed: {X_train.shape}, {X_val.shape}, {X_test.shape}")

Shapes confirmed: (1248, 56), (67, 56), (67, 56)


In [301]:
print(X_train.shape, y_train.shape)
print(X_val.shape, y_val.shape)
print(X_test.shape, y_test.shape)

(1248, 56) (1248,)
(67, 56) (67,)
(67, 56) (67,)


## Creating Logistic Regression Baseline Model

In [302]:
scaler = StandardScaler()
scaler.fit(X_train)
X_train_scaled = scaler.transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

In [303]:
model = LogisticRegression(max_iter=1000, random_state = 0)
model.fit(X_train_scaled, y_train)
linear_predictions = model.predict(X_val_scaled)



In [304]:
# Metrics
linear_accuracy = accuracy_score(y_val, linear_predictions)
linear_logloss = log_loss(y_val, model.predict_proba(X_val_scaled))
linear_brier_score = brier_score_loss(y_val,model.predict_proba(X_val_scaled)[:,1])

linear_accuracy, linear_logloss, linear_brier_score

(0.5970149253731343, 0.6563119844426238, 0.22070852194993965)

## XGBoost Testing

#### v1 (base values)

In [305]:
model_v1 = XGBClassifier(n_estimators=1000, learning_rate=0.05, early_stopping_rounds=20, max_depth=4, n_jobs=4, random_state=0)
model_v1.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
XGB_predictions = model_v1.predict(X_val)


In [306]:
XGB1_accuracy = accuracy_score(y_val, XGB_predictions)
XGB1_logloss = log_loss(y_val, model_v1.predict_proba(X_val))
XGB1_brier_score = brier_score_loss(y_val, model_v1.predict_proba(X_val)[:,1])

XGB1_accuracy, XGB1_logloss, XGB1_brier_score

(0.6865671641791045, 0.5946938401559722, 0.2073606252670288)

#### v2 (testing max_depth, best: depth=6)

In [307]:
model_v2 = XGBClassifier(n_estimators=1000, learning_rate=0.05, early_stopping_rounds=20, max_depth=6, n_jobs=4, random_state=0)
model_v2.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
XGB_predictions = model_v2.predict(X_val)


In [308]:
XGB2_accuracy = accuracy_score(y_val, XGB_predictions)
XGB2_logloss = log_loss(y_val, model_v2.predict_proba(X_val))
XGB2_brier_score = brier_score_loss(y_val, model_v2.predict_proba(X_val)[:,1])

XGB2_accuracy, XGB2_logloss, XGB2_brier_score

(0.7164179104477612, 0.5843532942765759, 0.20178599655628204)

(0.6417910447761194, 0.6108600782631547, 0.2133203148841858)
(0.6567164179104478, 0.6057593034597677, 0.2119840830564499)

#### v3 (testing learning_rate, best: lr=0.05)

In [309]:
model_v3 = XGBClassifier(n_estimators=1000, learning_rate=0.05, early_stopping_rounds=20, max_depth=6, n_jobs=4, random_state=0)
model_v3.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
XGB_predictions = model_v3.predict(X_val)

In [310]:
XGB3_accuracy = accuracy_score(y_val, XGB_predictions)
XGB3_logloss = log_loss(y_val, model_v3.predict_proba(X_val))
XGB3_brier_score = brier_score_loss(y_val, model_v3.predict_proba(X_val)[:,1])

XGB3_accuracy, XGB3_logloss, XGB3_brier_score

(0.7164179104477612, 0.5843532942765759, 0.20178599655628204)

#### v4 (testing subsample, best: subsample=0.8)

In [311]:
model_v4 = XGBClassifier(n_estimators=1000, learning_rate=0.05, early_stopping_rounds=20, max_depth=6, subsample = 0.8, n_jobs=4, random_state=0)
model_v4.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
XGB_predictions = model_v4.predict(X_val)

In [312]:
XGB4_accuracy = accuracy_score(y_val, XGB_predictions)
XGB4_logloss = log_loss(y_val, model_v4.predict_proba(X_val))
XGB4_brier_score = brier_score_loss(y_val, model_v4.predict_proba(X_val)[:,1])

XGB4_accuracy, XGB4_logloss, XGB4_brier_score

(0.7313432835820896, 0.5920516097214419, 0.20211507380008698)

### Final Model

In [313]:
final_model = model_v4
final_predictions = final_model.predict(X_test)
final_predictions
final_accuracy = accuracy_score(y_test, final_predictions)
final_logloss = log_loss(y_test, final_model.predict_proba(X_test))
final_brier_score = brier_score_loss(y_test, final_model.predict_proba(X_test)[:,1])

final_accuracy, final_logloss, final_brier_score

(0.6716417910447762, 0.6308326991833935, 0.21999667584896088)

In [314]:
## Interpreting Results
teams = pd.read_csv("../data/raw/Mteams.csv")
teams.head()
final_features = pd.read_parquet("../data/processed/final_features.parquet")


final_features = final_features[final_features['Season'] == 2024].reset_index(drop=True)
final_features = final_features.copy()
final_features.head()
assert len(final_features) == len(final_predictions), "Row count mismatch between final_features and predictions"
assert list(final_features.index) == list(X_test.reset_index(drop=True).index), "Index mismatch between final_features and X_test"
final_features['Predicted'] = final_predictions
final_features['WinProbability'] = final_model.predict_proba(X_test)[:,1]
final_features['Correct'] = final_features['Predicted'] == final_features['Results']

final_features.head()
final_features = pd.merge(final_features, teams, left_on='TeamA_Id', right_on='TeamID', how='left')
final_features = pd.merge(final_features, teams, left_on='TeamB_Id', right_on='TeamID', how='left')
final_results = final_features[['Season', 'DayNum', 'TeamA_Id', 'TeamB_Id', 'Predicted', 'WinProbability', 'Correct', 'TeamName_x', 'TeamName_y']]

conditions = [
    final_results['DayNum'] == 154,
    final_results['DayNum'] == 152,
    (final_results['DayNum'] >= 145) & (final_results['DayNum'] < 152),
    (final_results['DayNum'] >= 143) & (final_results['DayNum'] < 145),
    (final_results['DayNum'] >= 138) & (final_results['DayNum'] < 143),
    final_results['DayNum'] < 138
]

values = ["Championship", "Final Four", "Elite 8", "Sweet 16", "Round of 32", "Round of 64"]

final_results['Round'] = np.select(conditions, values, default="Unknown")

In [315]:
#Cleaning Up
final_results = final_results.drop(columns=['DayNum', 'TeamA_Id', 'TeamB_Id'])
final_results.rename(columns={'TeamName_x': 'TeamA', 'TeamName_y': 'TeamB', 'WinProbability': 'TeamA_WinProb', 'Predicted': 'Model_Pick'}, inplace=True)
final_results = final_results[['Season', 'Round', 'TeamA', 'TeamB', 'Model_Pick', 'TeamA_WinProb', 'Correct']]
final_results

,Season,Round,TeamA,TeamB,Model_Pick,TeamA_WinProb,Correct
0,2024,Round of 64,Virginia,Colorado St,0,0.435142,True
1,2024,Round of 64,Wagner,Howard,1,0.677526,True
2,2024,Round of 64,Colorado,Boise St,0,0.468784,False
3,2024,Round of 64,Grambling,Montana St,1,0.503516,True
4,2024,Round of 64,Long Beach St,Arizona,0,0.059070,True
5,2024,Round of 64,Creighton,Akron,1,0.587219,True
6,2024,Round of 64,Dayton,Nevada,0,0.385871,False
7,2024,Round of 64,Duquesne,BYU,0,0.109005,False
8,2024,Round of 64,McNeese St,Gonzaga,0,0.121744,True
9,2024,Round of 64,Illinois,Morehead St,1,0.518710,True
